# ARIMA Model

**Objective:** Build and train an Auto-ARIMA model to forecast January 2026 sales quantity and revenue, then benchmark against Prophet and LSTM.

**Input:** `monthly_aggregated.csv`  
**Currency:** LAK
**Training Period:** Jan 2023 – Sep 2025  
**Test Period:** Oct – Dec 2025  
**Forecast Target:** January 2026  

**What is ARIMA?**  
ARIMA stands for **AutoRegressive Integrated Moving Average** — a classical statistical time series model.  
- **AR (p):** Uses past values to predict future (like looking at last N months)  
- **I  (d):** Differencing to make the series stationary (removes trend)  
- **MA (q):** Uses past forecast errors to improve predictions  

**Why auto_arima?**  
Instead of manually guessing p, d, q — `auto_arima` tries all combinations and picks the one with the lowest **AIC (Akaike Information Criterion)** — a score that balances model accuracy vs complexity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pickle
import os

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded successfully')

In [ ]:
monthly = pd.read_csv('data/processed/monthly_aggregated.csv', parse_dates=['ds'])
monthly = monthly.sort_values('ds').reset_index(drop=True)

print(f'Loaded {len(monthly)} months of data')
print(f'Date range: {monthly["ds"].min().date()} to {monthly["ds"].max().date()}')
print()
monthly[['ds', 'total_qty', 'total_revenue']].tail(6)

### Train Test Split

Same split as Prophet and LSTM for a fair comparison:
- **Train:** Jan 2023 – Sep 2025 (33 months)
- **Test:** Oct 2025 – Dec 2025 (3 months)
- **Forecast:** Jan 2026

In [ ]:
TEST_MONTHS = 3

# Split
train = monthly.iloc[:-TEST_MONTHS]
test  = monthly.iloc[-TEST_MONTHS:]

# Extract series
qty_train = train['total_qty'].values
qty_test  = test['total_qty'].values

rev_train = train['total_revenue'].values
rev_test  = test['total_revenue'].values

print(f'Training months : {len(train)} (Jan 2023 – Sep 2025)')
print(f'Test months     : {len(test)} (Oct – Dec 2025)')

### Evaluation Functions

In [ ]:
def compute_metrics(actual, predicted, label):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100

    print(f'=== {label} — Test Set Evaluation (Oct-Dec 2025) ===')
    print(f'MAE  : {mae:,.2f}')
    print(f'RMSE : {rmse:,.2f}')
    print(f'MAPE : {mape:.2f}%')
    print()
    return mae, rmse, mape

print('Evaluation function defined')

## Auto ARIMA Model for Quantity

### Find Best ARIMA Order

`auto_arima` will try different combinations of p (0–5), d (0–2), q (0–5) and return the model with the lowest AIC score.

- `seasonal=True, m=12` → tells it to also look for seasonal patterns (yearly)
- `stepwise=True` → faster search using a smart stepwise algorithm instead of brute force
- `suppress_warnings=True` → keeps output clean

In [ ]:
print('Running auto_arima for Quantity — this may take 1-2 minutes...')
print()

arima_qty = auto_arima(
    qty_train,
    start_p=0, max_p=5,
    start_q=0, max_q=5,
    start_P=0, max_P=5,
    start_Q=0, max_Q=5,
    d=1,
    D=1,              # auto-detect differencing needed
    seasonal=True,
    m=12,                # monthly data, yearly seasonality
    stepwise=False,
    suppress_warnings=True,
    information_criterion='aic',
    error_action='ignore',
    trace=True           # prints each model it tries
)

print()
print(f'Best ARIMA order selected : {arima_qty.order}')
print(f'Seasonal order            : {arima_qty.seasonal_order}')
print(f'AIC Score                 : {arima_qty.aic():.2f}')

In [ ]:
print(arima_qty.summary())

### Evaluate on Test data for Quantity

In [ ]:
# Predict 3 months ahead
qty_pred_arima, qty_conf_int = arima_qty.predict(n_periods=TEST_MONTHS, return_conf_int=True)

# Metrics
mae_qty, rmse_qty, mape_qty = compute_metrics(qty_test, qty_pred_arima, 'QUANTITY')

# Month-by-month
print('Month-by-month comparison:')
for i, row in enumerate(test.itertuples()):
    err = (qty_pred_arima[i] - row.total_qty) / row.total_qty * 100
    print(f'  {row.ds.strftime("%b %Y")}: Actual={row.total_qty:,.0f} | Predicted={qty_pred_arima[i]:,.0f} | Error={err:+.1f}%')

print()
print(f'Prophet Quantity MAPE : 19.08%')
print(f'ARIMA   Quantity MAPE : {mape_qty:.2f}%')

### Forecast for Jan 2026 on Quantity

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

# Full historical
ax.plot(monthly['ds'], monthly['total_qty'],
        color='steelblue', linewidth=2, marker='o', markersize=4, label='Actual')

# Test predictions
ax.plot(test['ds'], qty_pred_arima,
        color='red', linewidth=2.5, marker='s', markersize=7, linestyle='--', label='ARIMA Predicted')

# Confidence interval
ax.fill_between(test['ds'], qty_conf_int[:, 0], qty_conf_int[:, 1],
                alpha=0.15, color='red', label='95% Confidence Interval')

# Highlight test region
ax.axvspan(test['ds'].iloc[0], test['ds'].iloc[-1], alpha=0.05, color='red')
ax.axvline(test['ds'].iloc[0], color='red', linestyle=':', alpha=0.5)
ax.text(test['ds'].iloc[0], monthly['total_qty'].max()*0.95, 'Test Period →',
        color='red', fontsize=9)

ax.set_title(f'ARIMA — Quantity: Actual vs Predicted (MAPE: {mape_qty:.2f}%)', fontweight='bold')
ax.set_ylabel('Quantity (Units)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()

plt.tight_layout()
plt.savefig('reports/11_arima_qty_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/11_arima_qty_forecast.png')

### Auto ARIMA for Revenue

In [ ]:
print('Running auto_arima for Revenue — this may take 1-2 minutes...')
print()

arima_rev = auto_arima(
    rev_train,
    start_p=0, max_p=5,
    start_q=0, max_q=5,
    start_P=0, max_P=5,
    start_Q=0, max_Q=5,
    d=1,
    D=1,
    seasonal=True,
    m=12,
    stepwise=False,
    suppress_warnings=True,
    information_criterion='aic',
    error_action='ignore',
    trace=True
)

print()
print(f'Best ARIMA order selected : {arima_rev.order}')
print(f'Seasonal order            : {arima_rev.seasonal_order}')
print(f'AIC Score                 : {arima_rev.aic():.2f}')

In [ ]:
print(arima_rev.summary())

### Evaluate on Test Dataset for Revenue

In [ ]:
# Predict 3 months ahead
rev_pred_arima, rev_conf_int = arima_rev.predict(n_periods=TEST_MONTHS, return_conf_int=True)

# Metrics
mae_rev, rmse_rev, mape_rev = compute_metrics(rev_test, rev_pred_arima, 'REVENUE')

# Month-by-month
print('Month-by-month comparison:')
for i, row in enumerate(test.itertuples()):
    err = (rev_pred_arima[i] - row.total_revenue) / row.total_revenue * 100
    print(f'  {row.ds.strftime("%b %Y")}: Actual=LAK {row.total_revenue:,.0f} | Predicted=LAK {rev_pred_arima[i]:,.0f} | Error={err:+.1f}%')

print()
print(f'Prophet Revenue MAPE : 11.92%')
print(f'ARIMA   Revenue MAPE : {mape_rev:.2f}%')

### Forecast on Jan 2026 for Revenue

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(monthly['ds'], monthly['total_revenue'],
        color='darkorange', linewidth=2, marker='o', markersize=4, label='Actual')

ax.plot(test['ds'], rev_pred_arima,
        color='red', linewidth=2.5, marker='s', markersize=7, linestyle='--', label='ARIMA Predicted')

ax.fill_between(test['ds'], rev_conf_int[:, 0], rev_conf_int[:, 1],
                alpha=0.15, color='red', label='95% Confidence Interval')

ax.axvspan(test['ds'].iloc[0], test['ds'].iloc[-1], alpha=0.05, color='red')
ax.axvline(test['ds'].iloc[0], color='red', linestyle=':', alpha=0.5)
ax.text(test['ds'].iloc[0], monthly['total_revenue'].max()*0.95, 'Test Period →',
        color='red', fontsize=9)

ax.set_title(f'ARIMA — Revenue: Actual vs Predicted (MAPE: {mape_rev:.2f}%)', fontweight='bold')
ax.set_ylabel('Revenue (LAK)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:,.0f}B'))
ax.legend()

plt.tight_layout()
plt.savefig('reports/12_arima_rev_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/12_arima_rev_forecast.png')

### Retrain on full 36 months to fully forecast for Jan 2026

In [ ]:
print('Retraining on full 36 months for Jan 2026 forecast...')

# Retrain quantity model on all 36 months
arima_qty.update(qty_test)
jan26_qty_arima, jan26_qty_conf = arima_qty.predict(n_periods=1, return_conf_int=True)

# Retrain revenue model on all 36 months
arima_rev.update(rev_test)
jan26_rev_arima, jan26_rev_conf = arima_rev.predict(n_periods=1, return_conf_int=True)

print()
print('=' * 55)
print('   ARIMA — JANUARY 2026 FORECAST')
print('=' * 55)
print(f'Quantity : {jan26_qty_arima[0]:,.0f} units')
print(f'          (range: {jan26_qty_conf[0][0]:,.0f} to {jan26_qty_conf[0][1]:,.0f})')
print()
print(f'Revenue  : LAK {jan26_rev_arima[0]:,.0f}')
print(f'Revenue  : LAK {jan26_rev_arima[0]/1e9:.2f} Billion')
print(f'          (range: LAK {jan26_rev_conf[0][0]/1e9:.2f}B to LAK {jan26_rev_conf[0][1]/1e9:.2f}B)')
print(f'          (range: LAK {jan26_rev_conf[0][0]:,.0f} to LAK {jan26_rev_conf[0][1]:,.0f})')
print()
print('--- Prophet January 2026 Forecast (for comparison) ---')
print(f'Quantity : 843,744 units  (range: 767,065 to 923,248)')
print(f'Revenue  : LAK 240.26 Billion')

### Save ARIMA Model

In [ ]:
os.makedirs('models/saved', exist_ok=True)

with open('models/saved/arima_qty_model.pkl', 'wb') as f:
    pickle.dump(arima_qty, f)

with open('models/saved/arima_rev_model.pkl', 'wb') as f:
    pickle.dump(arima_rev, f)

print('Models saved:')
print('  models/saved/arima_qty_model.pkl')
print('  models/saved/arima_rev_model.pkl')

### MODEL SUMMARY

In [ ]:
print('=' * 55)
print('   ARIMA MODEL SUMMARY')
print('=' * 55)
print()
print('MODEL CONFIGURATION:')
print(f'  Method              : Auto ARIMA (stepwise AIC selection)')
print(f'  Qty ARIMA Order     : {arima_qty.order}')
print(f'  Qty Seasonal Order  : {arima_qty.seasonal_order}')
print(f'  Rev ARIMA Order     : {arima_rev.order}')
print(f'  Rev Seasonal Order  : {arima_rev.seasonal_order}')
print(f'  Seasonality Period  : m=12 (monthly)')
print()
print('TEST SET PERFORMANCE (Oct-Dec 2025):')
print(f'  Quantity MAPE : {mape_qty:.2f}%  (Prophet: 19.08%)')
print(f'  Revenue  MAPE : {mape_rev:.2f}%  (Prophet: 11.92%)')
print()
print('JANUARY 2026 FORECAST:')
print(f'  Quantity : {jan26_qty_arima[0]:,.0f} units')
print(f'  Revenue  : LAK {jan26_rev_arima[0]:,.0f}')
print(f'  Revenue  : LAK {jan26_rev_arima[0]/1e9:.2f} Billion')